# Unit 2 exercises: instruments for the track

The starter steers at a point three waypoints ahead and never looks further.
Before any controller can do better, the car needs instruments: which way it is pointing relative to a target, where it is along the track, how far it is from the centreline, and how sharply the road bends ahead.
These exercises build each instrument, test it against something you can work out by hand, and then point it at the starter.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Later exercises reuse the functions you write in earlier ones, so work in order.
Keep any file you write inside `workdir`, a fresh temporary directory.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import CONTROL_DT, run_race
from racekit.tracks import load_track

STARTER = Path("../../projects/racekit/agents/unit-01.py")
workdir = Path(mkdtemp(prefix="roar-unit-02-ex-"))
deterministic = lambda: KinematicCar(jitter=0.0)
print("scratch files go in", workdir)

## Exercise 1 - The car's view of three points

A car is at (120.0, −40.0) with yaw 170° (in the world frame: x, y in metres, yaw counter-clockwise from +x).
Three points are at A = (100.0, −43.0), B = (118.0, −30.0) and C = (135.0, −42.0).

1. Write `wrap_angle(angle)`, wrapping radians to [−π, π), and `to_car_frame(px, py, cx, cy, yaw)`, returning `(forward, left)`.
2. Describe each point from the car.

Define:

- `car_frame`: a dict from `"A"`, `"B"`, `"C"` to its `(forward, left)` tuple (m);
- `errors_deg`: a dict from each name to the heading error to that point, in degrees, in [−180, 180): the angle the car would have to turn through, left positive;
- `naive_deg_A`: `atan2`'s bearing to A minus the yaw, in degrees, **without** wrapping;
- `behind`: the sorted list of the names of the points behind the car;
- `steer`: a dict from each name to the starter's steer toward that point at 20 m/s, `clip(-8 / sqrt(20) * error / pi, -1, 1)` with the error in radians (positive steers right);
- `which_way`: one sentence saying which way the starter turns for each point, and why `naive_deg_A` would have turned it the wrong way.

In [ ]:
car_x, car_y, car_yaw = 120.0, -40.0, math.radians(170.0)
POINTS = {"A": (100.0, -43.0), "B": (118.0, -30.0), "C": (135.0, -42.0)}

car_frame = {}  # TODO
errors_deg = {}  # TODO
naive_deg_A = None  # TODO
behind = []  # TODO
steer = {}  # TODO
which_way = "TODO"

## Exercise 2 - Differencing the yaw column

Race the starter on the `oval` with a deterministic car (`deterministic()`).
The yaw rate is how fast the heading changes: Δyaw / Δt between consecutive telemetry rows (Δt is `CONTROL_DT`).

Define:

- `naive_max_rate`: the largest |Δyaw| / Δt from the raw column, `np.diff(yaw) / CONTROL_DT` (rad/s);
- `wrapped_rates`: the array of yaw rates with each Δyaw wrapped first (rad/s);
- `wrapped_max_rate`: the largest absolute wrapped rate (rad/s);
- `wrap_jumps`: the number of consecutive-row pairs whose raw |Δyaw| exceeds π;
- `total_turn`: the sum of the wrapped Δyaw over the race (rad), and `laps_turned`: `total_turn / (2π)`;
- `bend_rate`: the median of |wrapped rate| over the rows where it exceeds 0.1 rad/s (the bends), and `bend_rate_prediction`: the yaw rate a car at a steady 19.49 m/s (Unit 1's settled speed) has on the oval's 120 m bends, v / R (rad/s);
- `why_jumps`: one sentence on why the raw column jumps so many times on this track, although the car never spins.

In [ ]:
oval_race = None  # TODO
naive_max_rate = None  # TODO
wrapped_rates = None  # TODO
wrapped_max_rate = None  # TODO
wrap_jumps = None  # TODO
total_turn = None  # TODO
laps_turned = None  # TODO
bend_rate = None  # TODO
bend_rate_prediction = None  # TODO
why_jumps = "TODO"

## Exercise 3 - Projection, and a stale hint

**(a) By hand, then in code.**
Write `project_onto_segment(px, py, ax, ay, bx, by)`, returning `(t, d2)`: the foot parameter t clamped to [0, 1], and the squared distance from p to the clamped foot.
For the segment a = (2, 1) → b = (10, 1), work out by hand, then check with your function:

- `t_raw_p`, `t_p`, `dist_p`: the **unclamped** t, the clamped t, and the distance to the clamped foot for p = (6, 5);
- `t_raw_q`, `t_q`, `dist_q`: the same for q = (−1, −3).

**(b) The windowed search.**
Write `nearest_segment(px, py, xs, ys, hint, reach=8)` with the lesson's rules (segments `hint − 8 … hint + 8`, cyclic; the smallest squared clamped-foot distance wins; ties to the first by strict `<`).
The setup below builds a point `(px, py)` on `test-circuit`: 30% of the way along segment 600, then 1 m to its left.
Search for it three times, with `hint` = 600, 0 and 585.

Define:

- `found`: a dict from each hint to the segment your search returns;
- `foot_distance`: a dict from each hint to the distance (m) from the point to that segment's clamped foot;
- `brute`: the segment a brute-force search over **all** segments returns;
- `silent_failure`: one sentence on what the search returns when the window does not contain the right segment, and how `foot_distance` lets you detect it.

In [ ]:
t_raw_p, t_p, dist_p = None, None, None  # TODO
t_raw_q, t_q, dist_q = None, None, None  # TODO

tc = load_track("test-circuit")
tc_x, tc_y = [float(value) for value in tc.x], [float(value) for value in tc.y]
dx, dy = tc_x[601] - tc_x[600], tc_y[601] - tc_y[600]
length = math.hypot(dx, dy)
px = tc_x[600] + 0.3 * dx - dy / length
py = tc_y[600] + 0.3 * dy + dx / length

found = {}  # TODO: hints 600, 0 and 585
foot_distance = {}  # TODO
brute = None  # TODO
silent_failure = "TODO"

## Exercise 4 - The cross-track error of the starter on `test-circuit`

Write `signed_offset(px, py, ax, ay, bx, by)` (the signed distance from p to the line through a → b, left positive, using the scalar cross product) and `cross_track(px, py, xs, ys, hint)` returning `(segment, offset)`.
Race the starter on `test-circuit` with `deterministic()`, and compute the cross-track error of every telemetry row, carrying the hint from 0 as racekit does.

Define:

- `tc_race`: the `RaceResult`;
- `my_offsets`: your array of cross-track errors, one per row, and `my_segments`: the matching segment indices;
- `max_disagreement`: the largest |`my_offsets` − the telemetry's `offset`| (m);
- `max_left`, `max_right`: the largest offset to the left (positive) and the most negative offset (m);
- `left_fraction`: the fraction of rows with a positive offset;
- `min_margin`: the smallest distance to the edge, `min(tc.half_width[my_segments] - |my_offsets|)` (m);
- `outside_of_corners`: `True` when, on every row whose segment has `|tc.curvature| > 0.01`, the offset and the generator's curvature have **opposite** signs.

In [ ]:
tc_race = None  # TODO
my_offsets, my_segments = None, None  # TODO
max_disagreement = None  # TODO
max_left, max_right = None, None  # TODO
left_fraction = None  # TODO
min_margin = None  # TODO
outside_of_corners = None  # TODO

## Exercise 5 - Curvature profiles, and a centimetre of noise

Write `menger_curvature(ax, ay, bx, by, cx, cy)` (signed, left turns positive) and `curvature_profile(xs, ys)` (one value per waypoint of a closed track, cyclic).

**(a) Noise.** Waypoints are 2 m apart.
Three points on a straight line, (−2, 0), (0, 0), (2, 0), have curvature 0.
Move the middle one 1 cm to the left, to (0, 0.01).
Define `kappa_bump` (1/m) and `radius_bump` (m), the radius of the "corner" that 1 cm of error invents.

**(b) Profiles.** For each of `"oval"`, `"test-circuit"` and `"monza-like"`, define in a dict keyed by track name:

- `r_min`: the smallest radius on the track (m);
- `straight_fraction`: the fraction of waypoints with |κ| < 0.001;
- `total_turning`: Σ κᵢ·dsᵢ over one lap, where dsᵢ is the length of segment i (rad);
- `direction`: `"counter-clockwise"` or `"clockwise"`, from the sign of `total_turning`;
- `corners`: the number of separate corners tighter than R = 100 m: maximal runs of consecutive waypoints (cyclically) with |κ| > 0.01 (a track may have none).

Then answer in `noise_lesson`: one sentence on what (a) means for estimating curvature from **measured** positions, such as a car's recorded path, rather than from the generator's clean waypoints.

In [ ]:
kappa_bump = None  # TODO
radius_bump = None  # TODO

r_min = {}  # TODO
straight_fraction = {}  # TODO
total_turning = {}  # TODO
direction = {}  # TODO
corners = {}  # TODO
noise_lesson = "TODO"

## Exercise 6 - Does a stiffer gain follow the line better, and is it faster?

On the `oval`, with `car_factory=lambda: KinematicCar(jitter=0.05)` and the default seeds 0–4, run the starter and a variant with `STEER_GAIN = 4.0` (made with `write_variant` into `workdir`).
For each race, compute the max and the RMS of |offset| over the driving rows (`held == 0`).

Define:

- `runs_8`, `runs_4`: the two `RepeatResult`s;
- `rms_8`, `rms_4`: lists of the five RMS values; `max_8`, `max_4`: lists of the five max values;
- `rms_median_8`, `rms_range_8`, `rms_median_4`, `rms_range_4`: their medians and ranges (m);
- `follows_better`: `noise_aware_better(rms_8, rms_4)` (lower is better, exactly as for times);
- `faster`: `noise_aware_better(runs_8.times, runs_4.times)`;
- `fig`, `ax`: one figure showing the seed-0 offset against `t` for both gains, with a shaded band (`fill_between`) of ± the gain-8 RMS median; a title, axis labels and a legend;
- `verdict`: one sentence on what the two verdicts say together.

In [ ]:
variant_4 = workdir / "gain_4.py"
runs_8, runs_4 = None, None  # TODO
rms_8, rms_4 = [], []  # TODO
max_8, max_4 = [], []  # TODO
rms_median_8, rms_range_8 = None, None  # TODO
rms_median_4, rms_range_4 = None, None  # TODO
follows_better = None  # TODO
faster = None  # TODO
fig, ax = None, None  # TODO
verdict = "TODO"

## Exercise 7 - Challenge: the sign of a corner, and the sign of the steer

Signed curvature is positive for a **left** turn.
The steer is positive for a **right** turn.
Check that the starter's actions respect the two conventions on `monza-like` (which runs clockwise), with your own instruments.

Race the starter on `monza-like` with `deterministic()`.
For each row, take the Menger curvature (your `curvature_profile`) at the segment your `cross_track` finds, and call the rows with |κ| > 0.01 the **corner rows**.

Define:

- `corner_rows`: the number of corner rows;
- `right_turn_fraction`: the fraction of corner rows with κ < 0;
- `opposite_sign_fraction`: the fraction of corner rows where `steer` and κ have opposite signs;
- `steer_per_kappa`: the median of `steer / κ` over the corner rows (m);
- `convention`: one or two sentences explaining why the opposite sign is the correct relation, and what `steer_per_kappa` tells you about how much steer a corner of radius R needs.
  (Unit 3 predicts this number from a model of the car; here you only measure it.)

In [ ]:
monza_race = None  # TODO
corner_rows = None  # TODO
right_turn_fraction = None  # TODO
opposite_sign_fraction = None  # TODO
steer_per_kappa = None  # TODO
convention = "TODO"

## Exercise 8 - Challenge: curvature and offset at 30 m/s

The lesson showed that the starter runs wide in every corner.
How tightly is the offset tied to the curvature, and does that change with speed?

Race the starter on `monza-like` with `deterministic()` at its own `TARGET_SPEED` of 20 m/s, and a variant at **30 m/s**.
For each race, with `my_segments` from your `cross_track`, correlate |offset| with |κ|, where κ is your Menger curvature at segment `(my_segments + shift) % n` (n waypoints), for every integer `shift` from −10 to 10.
Use the Pearson correlation, written with NumPy means and sums (no library correlation function):
r = Σ(a − ā)(b − b̄) / √(Σ(a − ā)² · Σ(b − b̄)²).

Define:

- `race_20`, `race_30`: the two `RaceResult`s, and `clean_30`: `True` when the 30 m/s race finished with 0 collisions;
- `r_at_car`: a dict from 20 and 30 to the correlation at `shift = 0`;
- `best_shift`: a dict from 20 and 30 to the shift with the highest correlation, and `r_best`: its correlation;
- `max_offset`: a dict from 20 and 30 to the max |offset| (m), and `offset_ratio`: `max_offset[30] / max_offset[20]`;
- `reading`: two sentences: what the sign of the best shift says about **when** the car runs wide, and whether 50% more speed changes the relation between curvature and offset or only its size.

In [ ]:
race_20, race_30 = None, None  # TODO
clean_30 = None  # TODO
r_at_car = {}  # TODO
best_shift = {}  # TODO
r_best = {}  # TODO
max_offset = {}  # TODO
offset_ratio = None  # TODO
reading = "TODO"